# AE와 frozen tail 탐지 중복 확인

동일 외부 구간 OOF 예측을 연결. 결과를 본 후 만든 OR이므로 독립적으로 선정/검증된 앙상블 아님. 다음 검증용 탐색 후보일 뿐. 참조 AE/꼬리점수 .05 설정은 동일 기준. CN7 단일 발생 이벤트 주의.


In [1]:
from pathlib import Path
import pandas as pd,numpy as np
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
a=pd.read_csv(t/'gas_normal_autoencoder_predictions.csv')
b=pd.read_csv(t/'gas_frozen_copula_predictions.csv')
a=a[(a.model=='normal_AE')&(a.budget==.05)]
b=b[(b.model=='frozen_COPOD_inspired')&(b.budget==.05)]
z=a.merge(b,on=['product','TimeStamp','block'],suffixes=('_ae','_tail'),validate='one_to_one')
assert len(z)==1325 and (z.actual_ae==z.actual_tail).all()
z['prediction_union']=np.maximum(z.prediction_ae,z.prediction_tail)
z['new_gas_vs_AE']=(z.actual_ae==1)&(z.prediction_tail==1)&(z.prediction_ae==0)
rows=[]
for product,g in z.groupby('product'):
 for name,col in [('AE','prediction_ae'),('tail','prediction_tail'),('exploratory_OR','prediction_union')]:
  tn,fp,fn,tp=confusion_matrix(g.actual_ae,g[col],labels=[0,1]).ravel()
  rows.append(dict(product=product,model=name,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn)))
m=pd.DataFrame(rows)
z.to_csv(t/'gas_tail_ae_exploratory_union_cases.csv',index=False,encoding='utf-8-sig')
m.to_csv(t/'gas_tail_ae_exploratory_union_metrics.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))
print(z[z.new_gas_vs_AE][['product','TimeStamp']].to_string(index=False))
print('Post-result exploratory union: not a preselected independently validated ensemble.')


product          model  TP  FP  FN  TN       F1
    CN7             AE   5  29   5 666 0.227273
    CN7           tail   6  34   4 661 0.240000
    CN7 exploratory_OR   6  42   4 653 0.206897
    RG3             AE   1  33  18 568 0.037736
    RG3           tail   0  12  19 589 0.000000
    RG3 exploratory_OR   1  38  18 563 0.034483
product           TimeStamp
    CN7 2020-10-16 05:45:26
Post-result exploratory union: not a preselected independently validated ensemble.
